# 1. Microstructure Research

流程: Orderbook 散点图 -> Wall Mid -> Normalized 图 -> Return ACF -> 交易分类 -> Adverse Selection

In [112]:
import sys
sys.path.insert(0, '.')

from utils.dataio import load_prices, load_prices_wide, load_trades
from utils.orderbook import compute_spread, return_autocorrelation, adverse_selection, order_interval_distribution, trade_interval_distribution, normalized_level_trade_stats
from utils.viz import plot_orderbook_scatter, plot_autocorrelation, plot_trade_profile, plot_normalized_orderbook, plot_interval_distribution
import polars as pl
import numpy as np

In [113]:
# === 配置 ===
ROUND = 1
DAY = -1
PRODUCT = 'ASH_COATED_OSMIUM'  # 改成你要分析的产品

# 逆向工程过滤开关（按量过滤）
FILTER_BY_VOLUME = False
MIN_ORDER_VOLUME = 10
MIN_TRADE_QUANTITY = 3

## Step 1: 加载数据

In [114]:
prices_long = load_prices(ROUND, DAY)
prices_wide = load_prices_wide(ROUND, DAY)
trades = load_trades(ROUND, DAY)

print('Products:', prices_long['product'].unique().to_list())
print('Prices shape:', prices_long.shape)
print('Trades shape:', trades.shape)
trades.head()

Products: ['ASH_COATED_OSMIUM', 'INTARIAN_PEPPER_ROOT']
Prices shape: (65263, 8)
Trades shape: (760, 7)


timestamp,buyer,seller,product,currency,price,quantity
i64,str,str,str,str,f64,i64
2800,null,null,"""ASH_COATED_OSMIUM""","""XIRECS""",9985.0,10
3000,null,null,"""INTARIAN_PEPPER_ROOT""","""XIRECS""",10999.0,8
4200,null,null,"""INTARIAN_PEPPER_ROOT""","""XIRECS""",10995.0,4
5200,null,null,"""ASH_COATED_OSMIUM""","""XIRECS""",10004.0,6
5200,null,null,"""ASH_COATED_OSMIUM""","""XIRECS""",9986.0,9


## Step 2: Orderbook 散点图（核心可视化）

In [115]:
# fair price 计算：同时生成外层 fair (outer) 与内层 fair (inner)
# outer: 用于挂单/库存估值；inner: 用于吃单/冲击判断（内层吸附到 +/-2）
REG_MIN_VOLUME = 15
VOL_THRESHOLD = 20.0
MAX_STALE_MS = 3000
HALF_SPREAD_CONST = 10.0      # 外层单边推断时使用固定半点差

# inner fair 参数（无 EMA，仅吸附）
INNER_PRIOR_OFFSET = -0.5     # 开头先验：inner = outer - 0.5
INNER_CONFLICT_TOL = 0.75     # 双侧候选偏移冲突阈值
INNER_OFFSET_MIN = -2.0
INNER_OFFSET_MAX = 1.0

wide_base = prices_wide.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in wide_base.columns:
    wide_base = wide_base.filter(pl.col('day') == DAY)


def _pick_max_vol_price(row: dict, side: str, vol_threshold: float = 20.0):
    """选择该侧订单量绝对值最大且大于阈值的报价。"""
    candidates = []
    for level in range(1, 4):
        p = row.get(f'{side}_price_{level}')
        v = row.get(f'{side}_volume_{level}')
        if p is None or v is None:
            continue
        v_abs = abs(float(v))
        if v_abs > vol_threshold:
            candidates.append((v_abs, float(p)))

    if not candidates:
        return None

    max_vol = max(v for v, _ in candidates)
    top_prices = [p for v, p in candidates if v == max_vol]
    if side == 'bid':
        return max(top_prices)
    return min(top_prices)


def _snap_half(x: float) -> float:
    return round(x * 2.0) / 2.0


def _adhere_offset_from_norm(px: float, norm_val: float, outer_fair: float):
    """
    outer-0.5 后，如果 norm 在 [+0.5, +3.5] 吸附到 +2；
    如果 norm 在 [-3.5, -0.5] 吸附到 -2。
    返回对应 offset（inner = outer + offset）。
    """
    if 0.5 <= norm_val <= 3.5:
        return px - 2.0 - outer_fair
    if -3.5 <= norm_val <= -0.5:
        return px + 2.0 - outer_fair
    return None


if PRODUCT == 'ASH_COATED_OSMIUM':
    ash = wide_base.sort('timestamp')

    ash_rows = []
    last_bid = None
    last_ask = None
    last_bid_ts = None
    last_ask_ts = None
    last_outer_fair = None

    for row in ash.iter_rows(named=True):
        ts = int(row['timestamp'])
        mid_now = float(row['mid_price']) if row.get('mid_price') is not None else None

        bid_obs = _pick_max_vol_price(row, 'bid', vol_threshold=VOL_THRESHOLD)
        ask_obs = _pick_max_vol_price(row, 'ask', vol_threshold=VOL_THRESHOLD)

        # 仅用历史信息的同侧缓存（有时效）
        bid_cache_ok = (last_bid is not None and last_bid_ts is not None and (ts - last_bid_ts) <= MAX_STALE_MS)
        ask_cache_ok = (last_ask is not None and last_ask_ts is not None and (ts - last_ask_ts) <= MAX_STALE_MS)

        use_bid = bid_obs if bid_obs is not None else (last_bid if bid_cache_ok else None)
        use_ask = ask_obs if ask_obs is not None else (last_ask if ask_cache_ok else None)

        outer_source = 'unavailable'
        outer_confidence = 0.0

        # A) 双边都有：最可信
        if use_bid is not None and use_ask is not None:
            outer_fair_px = (use_bid + use_ask) / 2.0
            outer_source = 'both_sides' if (bid_obs is not None and ask_obs is not None) else 'one_side_plus_cache'
            outer_confidence = 1.0 if outer_source == 'both_sides' else 0.8

        # B) 仅单边：用固定 half-spread 推断缺失侧
        elif use_ask is not None:
            use_bid = use_ask - 2.0 * HALF_SPREAD_CONST
            outer_fair_px = (use_bid + use_ask) / 2.0
            outer_source = 'infer_bid_from_ask'
            outer_confidence = 0.5

        elif use_bid is not None:
            use_ask = use_bid + 2.0 * HALF_SPREAD_CONST
            outer_fair_px = (use_bid + use_ask) / 2.0
            outer_source = 'infer_ask_from_bid'
            outer_confidence = 0.5

        # C) 双边都缺：优先 mid，其次沿用上一时刻 fair（仅前向）
        elif mid_now is not None:
            outer_fair_px = mid_now
            outer_source = 'mid_fallback'
            outer_confidence = 0.4

        elif last_outer_fair is not None:
            outer_fair_px = last_outer_fair
            outer_source = 'carry_fair'
            outer_confidence = 0.25

        else:
            outer_fair_px = None

        # inner fair（无 EMA：先 outer-0.5，再做区间吸附）
        inner_fair_px = None
        inner_offset = None
        inner_source = 'inner_unavailable'
        inner_confidence = 0.0

        if outer_fair_px is not None:
            baseline_offset = INNER_PRIOR_OFFSET
            baseline_inner_fair = outer_fair_px + baseline_offset

            bid1 = float(row['bid_price_1']) if row.get('bid_price_1') is not None else None
            ask1 = float(row['ask_price_1']) if row.get('ask_price_1') is not None else None
            bid1_vol = abs(float(row['bid_volume_1'])) if row.get('bid_volume_1') is not None else 1.0
            ask1_vol = abs(float(row['ask_volume_1'])) if row.get('ask_volume_1') is not None else 1.0

            offset_candidates = []
            offset_weights = []

            if bid1 is not None:
                bid1_norm = bid1 - baseline_inner_fair
                bid_off = _adhere_offset_from_norm(bid1, bid1_norm, outer_fair_px)
                if bid_off is not None:
                    offset_candidates.append(bid_off)
                    offset_weights.append(max(1.0, bid1_vol))

            if ask1 is not None:
                ask1_norm = ask1 - baseline_inner_fair
                ask_off = _adhere_offset_from_norm(ask1, ask1_norm, outer_fair_px)
                if ask_off is not None:
                    offset_candidates.append(ask_off)
                    offset_weights.append(max(1.0, ask1_vol))

            if len(offset_candidates) >= 2 and abs(offset_candidates[0] - offset_candidates[1]) > INNER_CONFLICT_TOL:
                target_offset = baseline_offset
                inner_source = 'inner_conflict_fallback'
                inner_confidence = 0.35
            elif len(offset_candidates) > 0:
                wsum = sum(offset_weights)
                target_offset = sum(v * w for v, w in zip(offset_candidates, offset_weights)) / wsum
                inner_source = 'inner_adhered_to_pm2'
                inner_confidence = 0.85
            else:
                target_offset = baseline_offset
                inner_source = 'inner_no_adhere_signal'
                inner_confidence = 0.55

            bounded_offset = min(max(target_offset, INNER_OFFSET_MIN), INNER_OFFSET_MAX)
            inner_offset = _snap_half(bounded_offset)
            inner_fair_px = outer_fair_px + inner_offset

        # 更新历史同侧缓存：仅记录“真实观测到”的侧
        if bid_obs is not None:
            last_bid = bid_obs
            last_bid_ts = ts
        if ask_obs is not None:
            last_ask = ask_obs
            last_ask_ts = ts
        if outer_fair_px is not None:
            last_outer_fair = outer_fair_px

        ash_rows.append({
            'day': row.get('day'),
            'timestamp': ts,
            'product': row['product'],
            'raw_mid': mid_now,
            'outer_wall_mid': outer_fair_px,
            'outer_source': outer_source,
            'outer_confidence': outer_confidence,
            'inner_wall_mid': inner_fair_px,
            'inner_offset': inner_offset,
            'inner_source': inner_source,
            'inner_confidence': inner_confidence,
            'bid_obs': bid_obs,
            'ask_obs': ask_obs,
            'bid_used': use_bid,
            'ask_used': use_ask,
            'half_spread_est': HALF_SPREAD_CONST,
        })

    fair_dual_df = pl.DataFrame(ash_rows).sort('timestamp')

    outer_wall_mid_df = fair_dual_df.select([
        'day', 'timestamp', 'product', 'raw_mid',
        pl.col('outer_wall_mid').alias('wall_mid'),
        pl.col('outer_source').alias('fair_source'),
        pl.col('outer_confidence').alias('fair_confidence'),
        'half_spread_est',
    ])

    inner_wall_mid_df = fair_dual_df.select([
        'day', 'timestamp', 'product', 'raw_mid',
        pl.col('inner_wall_mid').alias('wall_mid'),
        pl.col('inner_source').alias('fair_source'),
        pl.col('inner_confidence').alias('fair_confidence'),
        'inner_offset',
    ])

    # 默认 wall_mid_df 继续指向 outer，避免后续步骤行为突变
    wall_mid_df = outer_wall_mid_df

    print('ASH outer fair: online-only state machine (no backward fill), fixed half-spread=10')
    print('Outer fair source distribution:')
    print(outer_wall_mid_df.group_by('fair_source').agg(pl.len().alias('n')).sort('n', descending=True))

    print('\nASH inner fair: no-EMA adhesion after outer-0.5 shift')
    print('Adhesion rule: [+0.5,+3.5] -> +2, [-3.5,-0.5] -> -2')
    print('Inner fair source distribution:')
    print(inner_wall_mid_df.group_by('fair_source').agg(pl.len().alias('n')).sort('n', descending=True))

    inner_diag = (
        wide_base.select(['timestamp', 'product', 'bid_price_1', 'ask_price_1'])
        .join(inner_wall_mid_df.select(['timestamp', pl.col('wall_mid').alias('inner_fair')]), on='timestamp', how='left')
        .with_columns([
            (pl.col('bid_price_1') - pl.col('inner_fair')).alias('inner_bid_norm'),
            (pl.col('ask_price_1') - pl.col('inner_fair')).alias('inner_ask_norm'),
        ])
    )
    print('\nInner level-1 normalized diagnostics (vs inner fair):')
    print(inner_diag.select([
        pl.col('inner_bid_norm').drop_nulls().mean().alias('mean_bid_norm'),
        pl.col('inner_ask_norm').drop_nulls().mean().alias('mean_ask_norm'),
    ]))

else:
    # 非 ASH：外层沿用回归方案，内层采用 outer-0.5 固定偏移
    reg_src = prices_long.filter((pl.col('product') == PRODUCT) & (pl.col('volume') > REG_MIN_VOLUME))
    if DAY is not None and 'day' in reg_src.columns:
        reg_src = reg_src.filter(pl.col('day') == DAY)
    reg_src = reg_src.group_by('timestamp').agg(pl.col('price').mean().alias('reg_price')).sort('timestamp')

    if reg_src.height >= 2:
        x = reg_src['timestamp'].to_numpy()
        y = reg_src['reg_price'].to_numpy()
        k, b = np.polyfit(x, y, 1)
    else:
        k = 0.0
        mid_vals = wide_base['mid_price'].drop_nulls() if 'mid_price' in wide_base.columns else pl.Series([], dtype=pl.Float64)
        b = float(mid_vals.mean()) if len(mid_vals) else 0.0

    outer_wall_mid_df = wide_base.select(['day', 'timestamp', 'product', 'mid_price']).with_columns([
        pl.col('mid_price').alias('raw_mid'),
        (k * pl.col('timestamp') + b).round(0).alias('wall_mid'),
        pl.lit('regression').alias('fair_source'),
        pl.lit(0.6).alias('fair_confidence'),
        pl.lit(HALF_SPREAD_CONST).alias('half_spread_est'),
    ]).drop('mid_price')

    inner_wall_mid_df = outer_wall_mid_df.with_columns([
        (pl.col('wall_mid') + INNER_PRIOR_OFFSET).alias('wall_mid'),
        pl.lit('outer_minus_0_5').alias('fair_source'),
        pl.lit(0.5).alias('fair_confidence'),
        pl.lit(INNER_PRIOR_OFFSET).alias('inner_offset'),
    ])

    wall_mid_df = outer_wall_mid_df

    print(f'Outer fair regression (volume>{REG_MIN_VOLUME}): fair_price = {k:.8f} * timestamp + {b:.4f}')
    print('Inner fair (non-ASH): wall_mid_inner = wall_mid_outer - 0.5')

print('Preview: outer wall_mid_df')
display(outer_wall_mid_df.head())
print('Preview: inner wall_mid_df')
display(inner_wall_mid_df.head())

ASH outer fair: online-only state machine (no backward fill), fixed half-spread=10
Outer fair source distribution:
shape: (3, 2)
┌─────────────────────┬──────┐
│ fair_source         ┆ n    │
│ ---                 ┆ ---  │
│ str                 ┆ u32  │
╞═════════════════════╪══════╡
│ both_sides          ┆ 5772 │
│ one_side_plus_cache ┆ 4225 │
│ infer_bid_from_ask  ┆ 3    │
└─────────────────────┴──────┘

ASH inner fair: no-EMA adhesion after outer-0.5 shift
Adhesion rule: [+0.5,+3.5] -> +2, [-3.5,-0.5] -> -2
Inner fair source distribution:
shape: (2, 2)
┌────────────────────────┬──────┐
│ fair_source            ┆ n    │
│ ---                    ┆ ---  │
│ str                    ┆ u32  │
╞════════════════════════╪══════╡
│ inner_no_adhere_signal ┆ 9210 │
│ inner_adhered_to_pm2   ┆ 790  │
└────────────────────────┴──────┘

Inner level-1 normalized diagnostics (vs inner fair):
shape: (1, 2)
┌───────────────┬───────────────┐
│ mean_bid_norm ┆ mean_ask_norm │
│ ---           ┆ ---         

day,timestamp,product,raw_mid,wall_mid,fair_source,fair_confidence,half_spread_est
i64,i64,str,f64,f64,str,f64,f64
-1,0,"""ASH_COATED_OSMIUM""",10003.0,9993.0,"""infer_bid_from_ask""",0.5,10.0
-1,100,"""ASH_COATED_OSMIUM""",9992.0,9993.0,"""infer_bid_from_ask""",0.5,10.0
-1,200,"""ASH_COATED_OSMIUM""",9993.0,9993.0,"""infer_bid_from_ask""",0.5,10.0
-1,300,"""ASH_COATED_OSMIUM""",9993.0,9992.5,"""both_sides""",1.0,10.0
-1,400,"""ASH_COATED_OSMIUM""",9993.0,9993.0,"""one_side_plus_cache""",0.8,10.0


Preview: inner wall_mid_df


day,timestamp,product,raw_mid,wall_mid,fair_source,fair_confidence,inner_offset
i64,i64,str,f64,f64,str,f64,f64
-1,0,"""ASH_COATED_OSMIUM""",10003.0,9992.5,"""inner_no_adhere_signal""",0.55,-0.5
-1,100,"""ASH_COATED_OSMIUM""",9992.0,9992.5,"""inner_no_adhere_signal""",0.55,-0.5
-1,200,"""ASH_COATED_OSMIUM""",9993.0,9992.5,"""inner_no_adhere_signal""",0.55,-0.5
-1,300,"""ASH_COATED_OSMIUM""",9993.0,9992.0,"""inner_no_adhere_signal""",0.55,-0.5
-1,400,"""ASH_COATED_OSMIUM""",9993.0,9992.5,"""inner_no_adhere_signal""",0.55,-0.5


In [116]:
# Orderbook 散点图 + wall mid + trades
# 节选一个时间片段，避免一次性绘制全量数据导致内存压力
plot_start_ts = None      # 例如: 350000
plot_end_ts = None        # 例如: 450000
default_window = 300000   # 当 start/end 未指定时，默认只画前 100000ms

base_slice = prices_long.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in base_slice.columns:
    base_slice = base_slice.filter(pl.col('day') == DAY)

if base_slice.height > 0:
    ts_min = int(base_slice['timestamp'].min())
    ts_max = int(base_slice['timestamp'].max())
    if plot_start_ts is None:
        plot_start_ts = ts_min
    if plot_end_ts is None:
        plot_end_ts = min(ts_max, plot_start_ts + default_window)

print(f'Plot window: [{plot_start_ts}, {plot_end_ts}]')

fig = plot_orderbook_scatter(
    prices_long, trades, product=PRODUCT, day=DAY, wall_mid_df=wall_mid_df,
    start_ts=plot_start_ts,
    end_ts=plot_end_ts,
    filter_by_volume=FILTER_BY_VOLUME,
    min_order_volume=MIN_ORDER_VOLUME,
    min_trade_quantity=MIN_TRADE_QUANTITY,
    # use_resampler=True,  # 暂时注释掉重采样
    use_resampler=False,
)
fig.show()

Plot window: [0, 300000]


## Step 3: Normalized Orderbook（去趋势，Outer Fair vs Inner Fair）

In [117]:
print('Step 3A: Normalized by OUTER fair')
fig_outer = plot_normalized_orderbook(
    prices_long, outer_wall_mid_df, product=PRODUCT, day=DAY, trades=trades,
    filter_by_volume=FILTER_BY_VOLUME,
    min_order_volume=MIN_ORDER_VOLUME,
    min_trade_quantity=MIN_TRADE_QUANTITY,
    overlay_wall_mid_trend=True,
)
fig_outer.update_layout(title=f'{PRODUCT} Normalized Orderbook (Outer Fair)')
fig_outer.show()

print('Step 3B: Normalized by INNER fair')
fig_inner = plot_normalized_orderbook(
    prices_long, inner_wall_mid_df, product=PRODUCT, day=DAY, trades=trades,
    filter_by_volume=FILTER_BY_VOLUME,
    min_order_volume=MIN_ORDER_VOLUME,
    min_trade_quantity=MIN_TRADE_QUANTITY,
    overlay_wall_mid_trend=True,
)
fig_inner.update_layout(title=f'{PRODUCT} Normalized Orderbook (Inner Fair)')
fig_inner.show()

Step 3A: Normalized by OUTER fair


Step 3B: Normalized by INNER fair


## Step 4: Spread 统计

In [118]:
spread_df = compute_spread(prices_wide.filter(pl.col('product') == PRODUCT))
print('Wall spread stats:')
print(spread_df['wall_spread'].describe())
print(f"\nWall mid vs Raw mid diff: {(spread_df['wall_mid'] - spread_df['raw_mid']).mean():.4f}")

Wall spread stats:
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 9225.0    │
│ null_count ┆ 775.0     │
│ mean       ┆ 20.123469 │
│ std        ┆ 1.473103  │
│ min        ┆ 8.0       │
│ 25%        ┆ 19.0      │
│ 50%        ┆ 21.0      │
│ 75%        ┆ 21.0      │
│ max        ┆ 22.0      │
└────────────┴───────────┘

Wall mid vs Raw mid diff: 0.0188


## Step 5: Return Autocorrelation

In [119]:
wm = wall_mid_df.sort('timestamp')['wall_mid'].to_numpy()
acf, ci = return_autocorrelation(wm, max_lag=20)

fig = plot_autocorrelation(acf, ci, title=f'{PRODUCT} Return ACF')
fig.show()

# 解读
print(f'ACF(1) = {acf[0]:.4f}, 95% CI = +/-{ci:.4f}')
if abs(acf[0]) < ci:
    print('-> 不可预测，专注做市')
elif acf[0] < -ci:
    print('-> 短期均值回复，可以逆向交易')
else:
    print('-> 短期动量，taker 有 informed signal')

ACF(1) = -0.1426, 95% CI = +/-0.0196
-> 短期均值回复，可以逆向交易


## Step 6: 交易者分类

In [120]:
product_trades = trades.filter(pl.col('product') == PRODUCT)

if 'buyer' in trades.columns:
    fig = plot_trade_profile(trades, product=PRODUCT)
    fig.show()
    
    # 每个 trader 的行为模式
    for trader in product_trades['buyer'].unique().to_list():
        if not trader:
            continue
        tt = product_trades.filter(pl.col('buyer') == trader)
        print(f"\n{trader}: {len(tt)} buys, qty mode={tt['quantity'].mode().to_list()}, "
              f"avg_price={tt['price'].mean():.1f}")
else:
    print('No buyer/seller info in this round')

## Step 7: 订单间隔 / 成交间隔分布

In [121]:
order_intervals = order_interval_distribution(
    prices_long,
    product=PRODUCT,
    day=DAY,
    min_order_volume=MIN_ORDER_VOLUME if FILTER_BY_VOLUME else None,
)
trade_intervals = trade_interval_distribution(
    trades,
    product=PRODUCT,
    day=DAY,
    min_trade_quantity=MIN_TRADE_QUANTITY if FILTER_BY_VOLUME else None,
)

print(f"Order intervals count: {order_intervals.height}, mean={order_intervals['interval_ms'].mean() if order_intervals.height else 'NA'}")
print(f"Trade intervals count: {trade_intervals.height}, mean={trade_intervals['interval_ms'].mean() if trade_intervals.height else 'NA'}")

wm_overlay = wall_mid_df.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in wm_overlay.columns:
    wm_overlay = wm_overlay.filter(pl.col('day') == DAY)

fig_o = plot_interval_distribution(
    order_intervals,
    title=f"Order Intervals {PRODUCT} Day {DAY}",
    wall_mid_df=wm_overlay.select(['timestamp', 'wall_mid']),
)
fig_o.show()

fig_t = plot_interval_distribution(
    trade_intervals,
    title=f"Trade Intervals {PRODUCT} Day {DAY}",
    wall_mid_df=wm_overlay.select(['timestamp', 'wall_mid']),
)
fig_t.show()

Order intervals count: 9982, mean=100.17030655179323
Trade intervals count: 418, mean=2380.3827751196172


## Step 8: Wall Mid 奇偶统计

In [122]:
wm_stats = wall_mid_df.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in wm_stats.columns:
    wm_stats = wm_stats.filter(pl.col('day') == DAY)
wm_stats = wm_stats.filter(pl.col('wall_mid').is_not_null()).with_columns(
    (2*pl.col('wall_mid') % 2 == 0).alias('is_even')
)

ratio_tbl = wm_stats.group_by('is_even').agg(pl.len().alias('n')).with_columns(
    (pl.col('n') / pl.col('n').sum()).alias('ratio')
).sort('is_even')

print('2 * Wall mid 奇偶占比:')
print(ratio_tbl)

trades_with_wm = (
    trades.filter(pl.col('product') == PRODUCT)
    .join(wm_stats.select(['timestamp', 'product', 'wall_mid', 'is_even']), on=['timestamp', 'product'], how='left')
    .filter(pl.col('is_even').is_not_null())
)

qty_col = 'quantity' if 'quantity' in trades_with_wm.columns else ('volume' if 'volume' in trades_with_wm.columns else None)
if qty_col is None:
    print('Trades 中没有 quantity/volume 列，无法统计成交量。')
else:
    trade_qty_tbl = (
        trades_with_wm.group_by('is_even')
        .agg([
            pl.col(qty_col).abs().sum().alias('total_trade_qty'),
            pl.len().alias('trade_count'),
        ])
        .sort('is_even')
    )
    print('\nWall mid 奇偶时的总成交量:')
    print(trade_qty_tbl)

even_ratio = ratio_tbl.filter(pl.col('is_even') == True)['ratio'].item() if ratio_tbl.filter(pl.col('is_even') == True).height else 0.0
odd_ratio = ratio_tbl.filter(pl.col('is_even') == False)['ratio'].item() if ratio_tbl.filter(pl.col('is_even') == False).height else 0.0
print(f"\nEven ratio={even_ratio:.4f}, Odd ratio={odd_ratio:.4f}, Δfrom 50%={abs(even_ratio-0.5):.4f}")

2 * Wall mid 奇偶占比:
shape: (2, 3)
┌─────────┬──────┬────────┐
│ is_even ┆ n    ┆ ratio  │
│ ---     ┆ ---  ┆ ---    │
│ bool    ┆ u32  ┆ f64    │
╞═════════╪══════╪════════╡
│ false   ┆ 8966 ┆ 0.8966 │
│ true    ┆ 1034 ┆ 0.1034 │
└─────────┴──────┴────────┘

Wall mid 奇偶时的总成交量:
shape: (2, 3)
┌─────────┬─────────────────┬─────────────┐
│ is_even ┆ total_trade_qty ┆ trade_count │
│ ---     ┆ ---             ┆ ---         │
│ bool    ┆ i64             ┆ u32         │
╞═════════╪═════════════════╪═════════════╡
│ false   ┆ 1980            ┆ 380         │
│ true    ┆ 245             ┆ 45          │
└─────────┴─────────────────┴─────────────┘

Even ratio=0.1034, Odd ratio=0.8966, Δfrom 50%=0.3966


## Step 9: 标准化档位成交统计

In [123]:
level_stats = normalized_level_trade_stats(
    trades=trades,
    wall_mid_df=wall_mid_df,
    product=PRODUCT,
    day=DAY,
    round_to=None,
)

print('标准化档位成交统计（norm_level = trade_price - wall_mid）:')
print(level_stats)

if level_stats.height:
    print('\n按成交次数排序 Top10:')
    print(level_stats.sort('trade_count', descending=True).head(10))

# 验证：norm_level 是否符合 0.5 网格，以及 .0/.5 占比
norm = level_stats.select(['norm_level', 'trade_count']).with_columns([
    (pl.col('norm_level') * 2).round(8).alias('norm_x2'),
]).with_columns([
    ((pl.col('norm_x2') - pl.col('norm_x2').round(0)).abs() < 1e-8).alias('is_half_grid'),
    (pl.col('norm_x2').round(0) % 2 == 0).alias('is_integer_level'),
])

total_trades = norm['trade_count'].sum() if norm.height else 0
on_grid_trades = norm.filter(pl.col('is_half_grid'))['trade_count'].sum() if norm.height else 0
int_trades = norm.filter(pl.col('is_half_grid') & pl.col('is_integer_level'))['trade_count'].sum() if norm.height else 0
half_trades = norm.filter(pl.col('is_half_grid') & (~pl.col('is_integer_level')))['trade_count'].sum() if norm.height else 0

print(f"\nHalf-grid integrity: {on_grid_trades}/{total_trades} = {on_grid_trades/total_trades if total_trades else 0:.4f}")
print(f"Integer-level trades (.0): {int_trades} ({int_trades/total_trades if total_trades else 0:.4f})")
print(f"Half-level trades (.5): {half_trades} ({half_trades/total_trades if total_trades else 0:.4f})")

标准化档位成交统计（norm_level = trade_price - wall_mid）:
shape: (20, 4)
┌────────────┬─────────────┬────────────────┬──────────────┐
│ norm_level ┆ trade_count ┆ total_quantity ┆ avg_quantity │
│ ---        ┆ ---         ┆ ---            ┆ ---          │
│ f64        ┆ u32         ┆ i64            ┆ f64          │
╞════════════╪═════════════╪════════════════╪══════════════╡
│ -10.5      ┆ 30          ┆ 153            ┆ 5.1          │
│ -9.5       ┆ 1           ┆ 6              ┆ 6.0          │
│ -9.0       ┆ 1           ┆ 6              ┆ 6.0          │
│ -8.5       ┆ 69          ┆ 360            ┆ 5.217391     │
│ -8.0       ┆ 21          ┆ 118            ┆ 5.619048     │
│ …          ┆ …           ┆ …              ┆ …            │
│ 8.5        ┆ 75          ┆ 366            ┆ 4.88         │
│ 10.0       ┆ 1           ┆ 5              ┆ 5.0          │
│ 10.5       ┆ 25          ┆ 147            ┆ 5.88         │
│ 11.0       ┆ 4           ┆ 24             ┆ 6.0          │
│ 11.5       ┆ 2      

## Step 10: Adverse Selection 检验

In [124]:
wm_series = wall_mid_df.filter(pl.col('product') == PRODUCT).select(['timestamp', 'wall_mid'])
as_result = adverse_selection(product_trades, wm_series)

print('Adverse Selection (taker PnL after trade):')
for h, v in as_result.items():
    label = 'SAFE (taker=noise)' if v <= 0 else 'DANGER (taker=informed)'
    print(f'  h={h:3d} steps: {v:+.4f}  {label}')

Adverse Selection (taker PnL after trade):
  h=  1 steps: +0.0129  DANGER (taker=informed)
  h=  5 steps: +0.0282  DANGER (taker=informed)
  h= 10 steps: -0.0047  SAFE (taker=noise)
  h= 50 steps: +0.2121  DANGER (taker=informed)


## 结论

在这里写你的观察...

In [125]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox


def _pick_inner_fair_series():
    candidates = []

    if 'fair_dual_df' in globals():
        try:
            pdf = fair_dual_df.to_pandas() if hasattr(fair_dual_df, 'to_pandas') else fair_dual_df
            candidates.append(('fair_dual_df', pdf))
        except Exception as e:
            print('fair_dual_df not usable:', e)

    if 'inner_wall_mid_df' in globals():
        try:
            pdf = inner_wall_mid_df.to_pandas() if hasattr(inner_wall_mid_df, 'to_pandas') else inner_wall_mid_df
            candidates.append(('inner_wall_mid_df', pdf))
        except Exception as e:
            print('inner_wall_mid_df not usable:', e)

    for name, df in candidates:
        cols = [c for c in df.columns if 'inner' in str(c).lower() and ('fair' in str(c).lower() or 'mid' in str(c).lower())]
        if cols:
            col = cols[0]
            s = pd.to_numeric(df[col], errors='coerce').dropna()
            if len(s) > 50:
                return name, col, s

    raise ValueError('No suitable inner fair series found in current kernel. Please run upstream cells first.')

src_name, col_name, px = _pick_inner_fair_series()

# Use first difference as return proxy for price-level fair values.
ret = px.diff().dropna()
abs_ret = ret.abs()
sq_ret = ret.pow(2)

print(f'Using series: {src_name}.{col_name}')
print(f'n_obs(px)={len(px)}, n_obs(ret)={len(ret)}')
print(f'zero-return ratio={(ret == 0).mean():.4f}')

lags = [1, 5, 10, 20]
acf_abs = acf(abs_ret, nlags=max(lags), fft=True)
acf_sq = acf(sq_ret, nlags=max(lags), fft=True)

print('\nACF(abs returns):')
for L in lags:
    print(f'lag {L:>2}: {acf_abs[L]: .4f}')

print('\nACF(squared returns):')
for L in lags:
    print(f'lag {L:>2}: {acf_sq[L]: .4f}')

lb_abs = acorr_ljungbox(abs_ret, lags=lags, return_df=True)
lb_sq = acorr_ljungbox(sq_ret, lags=lags, return_df=True)

print('\nLjung-Box on abs returns:')
print(lb_abs)

print('\nLjung-Box on squared returns:')
print(lb_sq)

sig_abs = (lb_abs['lb_pvalue'] < 0.05).sum()
sig_sq = (lb_sq['lb_pvalue'] < 0.05).sum()

print('\nConclusion (5% level):')
if sig_abs >= 2 or sig_sq >= 2:
    print('Evidence supports volatility clustering in inner fair.')
else:
    print('No strong evidence of volatility clustering in inner fair.')

Using series: fair_dual_df.inner_wall_mid
n_obs(px)=10000, n_obs(ret)=9999
zero-return ratio=0.7078

ACF(abs returns):
lag  1:  0.0375
lag  5:  0.0142
lag 10:  0.0080
lag 20: -0.0002

ACF(squared returns):
lag  1:  0.0180
lag  5:  0.0108
lag 10:  0.0053
lag 20: -0.0024

Ljung-Box on abs returns:
      lb_stat  lb_pvalue
1   14.045835   0.000178
5   19.250226   0.001726
10  23.253252   0.009849
20  28.566051   0.096662

Ljung-Box on squared returns:
      lb_stat  lb_pvalue
1    3.223624   0.072583
5    7.919587   0.160723
10  11.207415   0.341589
20  16.295085   0.698157

Conclusion (5% level):
Evidence supports volatility clustering in inner fair.


In [126]:
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox

ret_nz = ret[ret != 0]
abs_ret_nz = ret_nz.abs()
sq_ret_nz = ret_nz.pow(2)

print(f'n_obs(non-zero ret)={len(ret_nz)}')

lags = [1, 5, 10, 20]
lb_abs_nz = acorr_ljungbox(abs_ret_nz, lags=lags, return_df=True)
lb_sq_nz = acorr_ljungbox(sq_ret_nz, lags=lags, return_df=True)

print('\nLjung-Box on abs returns (non-zero only):')
print(lb_abs_nz)

print('\nLjung-Box on squared returns (non-zero only):')
print(lb_sq_nz)

# ARCH LM test on original return series
arch_lm = het_arch(ret.dropna(), nlags=10)
print('\nARCH-LM (ret, 10 lags):')
print(f'LM stat={arch_lm[0]:.4f}, LM pvalue={arch_lm[1]:.6f}, F stat={arch_lm[2]:.4f}, F pvalue={arch_lm[3]:.6f}')

n_obs(non-zero ret)=2922

Ljung-Box on abs returns (non-zero only):
       lb_stat     lb_pvalue
1   230.789692  4.009822e-52
5   273.240749  5.635360e-57
10  285.209988  2.070296e-55
20  294.622430  1.011786e-50

Ljung-Box on squared returns (non-zero only):
       lb_stat     lb_pvalue
1   170.801696  4.943909e-39
5   199.685420  3.316718e-41
10  208.435546  2.800140e-39
20  217.049295  4.629654e-35

ARCH-LM (ret, 10 lags):
LM stat=11.0813, LM pvalue=0.351217, F stat=1.1081, F pvalue=0.351342


In [127]:
# Step X: fair 上下方 × 理性/非理性 的订单次数/订单量到达率（8项）
# 统一只用 tick 口径：每个 timestamp 视为 1 tick。

fair_src = wall_mid_df.select(['timestamp', 'product', 'wall_mid'])

orders = prices_long.filter(pl.col('product') == PRODUCT)
if DAY is not None and 'day' in orders.columns:
    orders = orders.filter(pl.col('day') == DAY)

if FILTER_BY_VOLUME:
    orders = orders.filter(pl.col('volume').abs() >= MIN_ORDER_VOLUME)

side_expr = (
    pl.col('side')
    if 'side' in orders.columns
    else pl.when(pl.col('volume') > 0).then(pl.lit('bid')).otherwise(pl.lit('ask'))
)

orders = (
    orders
    .join(
        fair_src.filter(pl.col('product') == PRODUCT).select(['timestamp', 'wall_mid']),
        on='timestamp',
        how='left',
    )
    .filter(pl.col('wall_mid').is_not_null())
    .with_columns([
        (pl.col('price') - pl.col('wall_mid')).alias('delta'),
        side_expr.alias('side_used'),
    ])
    .filter(pl.col('delta') != 0)
    .with_columns([
        pl.when(pl.col('delta') > 0).then(pl.lit('above')).otherwise(pl.lit('below')).alias('position'),
        (
            ((pl.col('side_used') == 'bid') & (pl.col('delta') < 0)) |
            ((pl.col('side_used') == 'ask') & (pl.col('delta') > 0))
        ).alias('is_rational')
    ])
    .with_columns([
        pl.when(pl.col('is_rational')).then(pl.lit('rational')).otherwise(pl.lit('irrational')).alias('rationality'),
        pl.col('volume').abs().alias('abs_volume'),
    ])
)

ts_count = orders.select(pl.col('timestamp').n_unique().alias('n_ts')).item()
if ts_count == 0:
    print('没有可用订单（可能过滤条件过严或 fair 对齐后为空）。')
else:
    grouped = (
        orders
        .group_by(['position', 'rationality'])
        .agg([
            pl.len().alias('order_count'),
            pl.col('abs_volume').sum().alias('total_volume'),
        ])
        .with_columns([
            (pl.col('order_count') / ts_count).alias('count_rate_per_tick'),
            (pl.col('total_volume') / ts_count).alias('volume_rate_per_tick'),
        ])
    )

    keys = [
        ('above', 'rational'),
        ('above', 'irrational'),
        ('below', 'rational'),
        ('below', 'irrational'),
    ]

    print(f'PRODUCT={PRODUCT}, DAY={DAY}, tick样本数={ts_count}')
    print('--- 8项结果（position × rationality × tick口径）---')

    for pos, rat in keys:
        sub = grouped.filter((pl.col('position') == pos) & (pl.col('rationality') == rat))
        if sub.height == 0:
            c_tick = 0.0
            v_tick = 0.0
        else:
            c_tick = float(sub['count_rate_per_tick'][0])
            v_tick = float(sub['volume_rate_per_tick'][0])

        print(f'{pos:>5} | {rat:>10} | count/tick={c_tick:.6f}')
        print(f'{pos:>5} | {rat:>10} | volume/tick={v_tick:.6f}')

PRODUCT=ASH_COATED_OSMIUM, DAY=-1, tick样本数=9983
--- 8项结果（position × rationality × tick口径）---
above |   rational | count/tick=1.627968
above |   rational | volume/tick=30.124912
above | irrational | count/tick=0.013523
above | irrational | volume/tick=0.094060
below |   rational | count/tick=1.623861
below |   rational | volume/tick=30.021036
below | irrational | count/tick=0.012722
below | irrational | volume/tick=0.085045
